# 10 - Ensemble: Stacking (meta-model)

**Stacking** trains a second model (the *meta-model*) whose inputs are the base models' predicted probabilities. Instead of us choosing weights, the meta-model *learns* how to combine them - and it can give the weak models (logreg, knn) a role if they add a useful signal on certain rows.

Why it doesn't trivially overfit: we feed the meta-model the **out-of-fold** probabilities (each row predicted by base models that didn't train on it), and we evaluate the meta-model with the **same 5-fold CV**. A deliberately simple meta-model (logistic regression) further limits overfitting.

Output: `outputs/ensemble_stacking_submission.csv`.

In [1]:
import sys, os
sys.path.insert(0, os.getcwd())   # make the local helper importable
import numpy as np, pandas as pd
from sklearn.metrics import f1_score, classification_report
import ensemble_lib as eb        # shared: tuned base models + cached OOF/test probabilities

# Load each base model's probabilities. The FIRST run computes & caches them to
# outputs/preds/*.npy (slow, a few minutes); afterwards this is instant.
NAMES = ['svm', 'catboost', 'histgbm', 'logreg', 'knn']
oof  = {n: eb.get_proba(n)[0] for n in NAMES}   # leak-free train probabilities
tst  = {n: eb.get_proba(n)[1] for n in NAMES}   # test probabilities
print('\nIndividual OOF macro-F1 (baseline to beat):')
for n in NAMES:
    print(f'  {n:9s} {eb.macro_f1(oof[n]):.4f}')

[svm] loaded cached probabilities
[catboost] loaded cached probabilities
[histgbm] loaded cached probabilities
[logreg] loaded cached probabilities
[knn] loaded cached probabilities
[svm] loaded cached probabilities
[catboost] loaded cached probabilities
[histgbm] loaded cached probabilities
[logreg] loaded cached probabilities
[knn] loaded cached probabilities

Individual OOF macro-F1 (baseline to beat):
  svm       0.8324
  catboost  0.8216
  histgbm   0.8165
  logreg    0.7453
  knn       0.7641


## Step 1 - Build meta-features
Stack each model's 4 class-probabilities side by side: 5 models x 4 classes = 20 meta-features per row. Train uses OOF probabilities; test uses the base models' test probabilities.

In [2]:
STACK = ['svm', 'catboost', 'histgbm', 'logreg', 'knn']
X_meta_oof  = np.hstack([oof[m] for m in STACK])     # (9000, 20)
X_meta_test = np.hstack([tst[m] for m in STACK])     # (5000, 20)
print('meta-feature matrix:', X_meta_oof.shape, '->', len(STACK), 'models x 4 classes')

meta-feature matrix: (9000, 20) -> 5 models x 4 classes


## Step 2 - Evaluate the meta-model with CV
We use `cross_val_predict` on the meta-features to get honest stacked OOF predictions, then score them. This is our estimate of the stacking macro-F1.

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_predict

meta = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=1.0))
meta_oof = cross_val_predict(meta, X_meta_oof, eb.y, cv=eb.cv, method='predict_proba', n_jobs=-1)
print('stacking OOF macro-F1 = %.4f' % eb.macro_f1(meta_oof))
print('best single model     = %.4f' % max(eb.macro_f1(oof[n]) for n in NAMES))

stacking OOF macro-F1 = 0.8282
best single model     = 0.8324


## Step 3 - Per-class threshold tuning on the stacked OOF

In [4]:
cw, tuned = eb.tune_class_weights(meta_oof)
print('class weights:', np.round(cw, 3))
print('stacking OOF before class-tuning: %.4f' % eb.macro_f1(meta_oof))
print('stacking OOF after  class-tuning: %.4f' % tuned)
print(classification_report(eb.y, (meta_oof * cw).argmax(1), digits=3))

class weights: [1.1  1.25 1.4  1.  ]
stacking OOF before class-tuning: 0.8282
stacking OOF after  class-tuning: 0.8324
              precision    recall  f1-score   support

           0      0.869     0.854     0.861      2001
           1      0.845     0.857     0.851      2442
           2      0.771     0.798     0.784      2237
           3      0.848     0.819     0.833      2320

    accuracy                          0.832      9000
   macro avg      0.833     0.832     0.832      9000
weighted avg      0.833     0.832     0.832      9000



## Step 4 - Fit meta-model on all data and predict test
We refit the meta-model on the full OOF matrix, apply it to the test meta-features, apply the tuned class weights, and save.

In [5]:
meta.fit(X_meta_oof, eb.y)
meta_test = meta.predict_proba(X_meta_test)
final = (meta_test * cw).argmax(1).astype(int)

os.makedirs('../outputs', exist_ok=True)
sub = pd.DataFrame({'id': eb.test['id'], 'sleep_stage': final})
sub.to_csv('../outputs/ensemble_stacking_submission.csv', index=False)
print('wrote ../outputs/ensemble_stacking_submission.csv', sub.shape)
print('predicted class distribution:', dict(sub.sleep_stage.value_counts().sort_index()))
sub.head()

wrote ../outputs/ensemble_stacking_submission.csv (5000, 2)
predicted class distribution: {0: np.int64(1103), 1: np.int64(1305), 2: np.int64(1335), 3: np.int64(1257)}


,id,sleep_stage
0,9000,0
1,9001,3
2,9002,1
3,9003,2
4,9004,3


## Step 5 - What did the meta-model rely on?
Inspect the meta-model's coefficients to see which base models drive the final decision (larger magnitude = more influence).

In [6]:
lr = meta.named_steps['logisticregression']
coef_strength = np.abs(lr.coef_).sum(0).reshape(len(STACK), 4).sum(1)  # total |coef| per base model
infl = pd.Series(coef_strength, index=STACK).sort_values(ascending=False)
print('relative influence of each base model in the meta-model:')
print((infl / infl.sum()).round(3))

relative influence of each base model in the meta-model:
catboost    0.456
logreg      0.172
knn         0.166
svm         0.122
histgbm     0.083
dtype: float64


## Takeaways
- If stacking OOF > the weighted-vote OOF, the meta-model found a better-than-linear combination.
- The influence table usually shows svm + catboost dominating, with logreg/knn contributing small corrections - exactly the "weak models earn a small role" idea.
- Guard against overfitting: trust the OOF number, keep the meta-model simple, and compare the public-LB movement of all three ensembles before choosing your final submission.